# 14.17 使用時沒有正解，怎麼知道該停？


剛才我們畫 loss 曲線，是因為評估題有正解。真正來了一道新題，服務拿不到正解，就不能現算每輪 CE，再挑最低的一輪。那樣的停止器偷偷使用了評估答案。

最簡單的合法做法，是在 validation 上選一個固定輪數並凍結。另一個做法是讓使用者選 low／medium／high，映射到已驗過的輪數。兩者都不需要每題知道何時最好，但平均上可能多算或少算；它們應保留為 Auto 的比較基線。

要逐題選，可以在開始前讀問題特徵，預測預算；也可以每輪檢查可觀測訊號，例如最大預測機率、前後答案是否一致、狀態改變量。這些是**代理訊號**：能觀察到它們，不等於知道答案對。連續兩輪都選錯同一類，答案就已穩定；模型也可能錯得非常肯定。

下面是人工機率序列。兩輪 argmax 都是類別0，但真解是類別1；真解只供本例離線核對，部署停止規則不會得到它。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

predictions = [torch.tensor([0.95, 0.05]), torch.tensor([0.97, 0.03])]
for r, p in enumerate(predictions, start=1):
    print(r, "選類別", int(p.argmax()), "最大機率", float(p.max()))
print("離線核對用的真解類別", 1)

所以「機率超過0.9就停」在這個例子立即錯。用較高門檻不能解決所有失準，需要獨立校準與停止規則評估，記下每個規則答錯幾題、用了幾輪及多少時間。也可以學一個停止機率並在訓練中加入計算代價；ACT（Adaptive Computation Time，自適應計算時間）是早期研究之一。完整學習停止器還涉及離散動作或可微近似，本課列為進階，不要求學生先實作才能理解固定輪數。

服務第一版的 Auto 因而先做**開始前選預算**。先用判斷器看題目與資訊充分性，再依驗證過的規則選輪數；不假裝能讀到隱藏真解。若之後改成 low 失敗再重跑 high，兩次工作都要記成本；若想接續同一隱藏狀態，也要模型與快取協定真的支援。

練習說明「先看過真解再選最好輪」有什麼用途。它可作固定評估紀錄上的回看上界，提示尚有多少選擇空間；不代表一個能部署、免費且一定找得到好輪數的控制器。

來源：[ACT，2016](https://arxiv.org/abs/1603.08983)。
